# 🎯 Objectif : Générer des quiz JSON sur les habiletés sociales
# à partir de situations manuelles ou de chunks issus du corpus RAG.
# Chaque quiz inclut : contexte, options A/B/C, bonne réponse, commentaire doux.


In [15]:
# Script pour générer des quiz à choix multiples en utilisant l'API OpenAI

# Importations nécessaires
import os
import json
from openai import OpenAI  # Nouvelle syntaxe OpenAI v1.x
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

# Initialisation du client OpenAI (nouvelle API)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Dossier de sortie
output_dir = Path("generated_quizzes")
output_dir.mkdir(exist_ok=True)

In [16]:
# Fonction pour nettoyer les guillemets typographiques
def clean_quotes(text: str) -> str:
    return text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")


In [20]:
# Fonction pour générer un quiz à partir d'une situation donnée

def generate_quiz_json(situation_text: str) -> dict:
    system_prompt = """Tu es un assistant bienveillant qui crée des quiz sur les habiletés sociales pour adolescents autistes. 

IMPORTANT: Tu dois UNIQUEMENT retourner un JSON valide, sans texte avant ou après. 

Format JSON requis:
{
  "contexte": "Description de la situation",
  "question": "Que fais-tu dans cette situation ?",
  "options": {
    "A": "Première option",
    "B": "Deuxième option", 
    "C": "Troisième option"
  },
  "bonne_reponse": "A",
  "commentaire": "Explication bienveillante de pourquoi cette réponse est appropriée"
}"""
    
    user_prompt = f"Crée un quiz JSON pour cette situation: {situation_text}"
    
    # Nouvelle syntaxe OpenAI v1.x
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7
    )
    
    content = response.choices[0].message.content
    print(f"🔍 Réponse brute de GPT-4o:")
    print(f"'{content}'")
    print(f"🔍 Longueur: {len(content)} caractères")
    
    # Nettoyer le contenu
    cleaned = clean_quotes(content.strip())
    
    # Extraire le JSON si entouré de markdown
    if cleaned.startswith("```json"):
        cleaned = cleaned.replace("```json", "").replace("```", "").strip()
    elif cleaned.startswith("```"):
        cleaned = cleaned.replace("```", "").strip()
    
    print(f"🔍 Contenu nettoyé:")
    print(f"'{cleaned}'")
    
    try:
        quiz_json = json.loads(cleaned)
        print("✅ JSON parsé avec succès!")
        return quiz_json
    except json.JSONDecodeError as e:
        print(f"⚠️ Erreur de parsing JSON à la position {e.pos}: {e.msg}")
        print("📝 Contenu problématique:")
        print(cleaned)
        return {}

In [21]:
situation = (
    "Tu es à la cantine et tu cherches une place. Tu vois une personne seule. "
    "Tu aimerais te joindre à elle. Que fais-tu ?"
)

quiz = generate_quiz_json(situation)
print(json.dumps(quiz, indent=2, ensure_ascii=False))


🔍 Réponse brute de GPT-4o:
'```json
{
  "contexte": "Tu es à la cantine et tu cherches une place. Tu vois une personne seule. Tu aimerais te joindre à elle.",
  "question": "Que fais-tu dans cette situation ?",
  "options": {
    "A": "Tu t'approches et demandes poliment si tu peux t'asseoir avec elle.",
    "B": "Tu t'assois sans rien dire, en espérant qu'elle soit d'accord.",
    "C": "Tu cherches une autre table, même si tu préfères t'asseoir avec elle."
  },
  "bonne_reponse": "A",
  "commentaire": "Demander poliment avant de s'asseoir montre du respect pour l'espace de l'autre personne et ouvre la possibilité d'une interaction positive. Cela permet aussi à l'autre personne de te répondre et de créer un échange amical."
}
```'
🔍 Longueur: 711 caractères
🔍 Contenu nettoyé:
'{
  "contexte": "Tu es à la cantine et tu cherches une place. Tu vois une personne seule. Tu aimerais te joindre à elle.",
  "question": "Que fais-tu dans cette situation ?",
  "options": {
    "A": "Tu t'approch

In [22]:
def save_quiz(quiz_data: dict, filename: str):
    with open(output_dir / filename, "w", encoding="utf-8") as f:
        json.dump(quiz_data, f, indent=2, ensure_ascii=False)
    print(f"✅ Quiz sauvegardé dans {filename}")

save_quiz(quiz, "quiz_cantine.json")


✅ Quiz sauvegardé dans quiz_cantine.json


In [26]:
# 🎯 Génération automatisée de 20+ quiz sur diverses situations sociales

# Base de situations sociales variées pour adolescents autistes
situations_base = [
    # Situations scolaires
    "Tu arrives en retard en cours et tout le monde te regarde. Comment entres-tu dans la classe ?",
    "Ton professeur te demande de faire un exposé oral devant toute la classe. Tu es très stressé(e). Que fais-tu ?",
    "Un camarade copie sur toi pendant un contrôle. Tu ne sais pas quoi faire.",
    "Tu ne comprends pas un exercice et le professeur semble occupé. Comment demandes-tu de l'aide ?",
    "Tes camarades font du bruit pendant que tu essaies de te concentrer. Comment réagis-tu ?",
    
    # Situations sociales avec les pairs
    "Un groupe d'amis discute de quelque chose qui t'intéresse, mais tu n'oses pas t'approcher. Que fais-tu ?",
    "Quelqu'un se moque de tes centres d'intérêt devant d'autres personnes. Comment réagis-tu ?",
    "Tu veux inviter quelqu'un à ton anniversaire mais tu as peur qu'il/elle refuse. Comment procèdes-tu ?",
    "Tes amis planifient une sortie qui ne t'intéresse pas vraiment. Que leur dis-tu ?",
    "Tu entends des rumeurs sur toi qui sont fausses. Comment gères-tu cette situation ?",
    
    # Situations familiales
    "Tes parents te demandent de ranger ta chambre alors que tu es concentré(e) sur ton activité favorite. Comment réagis-tu ?",
    "Tu veux expliquer à tes parents pourquoi certains bruits ou textures te dérangent. Comment t'y prends-tu ?",
    "Ta famille organise un grand repas avec beaucoup de monde et de bruit. Tu te sens dépassé(e). Que fais-tu ?",
    
    # Situations en public
    "Tu es dans un magasin et tu ne trouves pas ce que tu cherches. Comment demandes-tu de l'aide au vendeur ?",
    "Dans les transports en commun, quelqu'un écoute de la musique très fort. Cela te dérange beaucoup. Que fais-tu ?",
    "Tu es dans une file d'attente et quelqu'un passe devant toi sans faire exprès. Comment réagis-tu ?",
    "Tu es perdu(e) dans un centre commercial et tu dois demander ton chemin. Comment procèdes-tu ?",
    
    # Situations de stress/conflit
    "Tu as un désaccord avec ton meilleur ami sur quelque chose d'important pour toi. Comment abordes-tu la discussion ?",
    "Quelqu'un critique tes habitudes ou tes routines. Tu te sens blessé(e). Que fais-tu ?",
    "Tu es dans une situation sociale bruyante et stimulante qui devient difficile à supporter. Comment gères-tu cela ?",
    
    # Situations d'entraide
    "Tu vois quelqu'un qui semble triste ou isolé. Tu aimerais l'aider mais tu ne sais pas comment t'y prendre.",
    "Un camarade te demande de l'aide pour quelque chose que tu maîtrises bien. Comment l'aides-tu efficacement ?"
]

print(f"📋 {len(situations_base)} situations prêtes à être transformées en quiz !")

📋 22 situations prêtes à être transformées en quiz !


In [27]:
# 🚀 Fonction pour générer tous les quiz automatiquement

def generate_all_quizzes(situations_list: list, batch_size: int = 5):
    """
    Génère tous les quiz et les sauvegarde avec une pause entre les batches
    pour éviter les limitations de taux de l'API OpenAI
    """
    import time
    
    all_quizzes = []
    total_situations = len(situations_list)
    
    print(f"🎯 Génération de {total_situations} quiz en cours...")
    print(f"📦 Traitement par batch de {batch_size} quiz")
    
    for i, situation in enumerate(situations_list):
        print(f"\n📝 Quiz {i+1}/{total_situations}: Génération en cours...")
        
        try:
            # Génération du quiz
            quiz = generate_quiz_json(situation)
            
            if quiz:  # Si le quiz a été généré avec succès
                # Sauvegarde individuelle
                filename = f"quiz_{i+1:02d}_auto.json"
                save_quiz(quiz, filename)
                all_quizzes.append(quiz)
                print(f"✅ Quiz {i+1} terminé !")
            else:
                print(f"❌ Échec pour le quiz {i+1}")
                
        except Exception as e:
            print(f"❌ Erreur pour le quiz {i+1}: {e}")
            
        # Pause entre les batches pour respecter les limites API
        if (i + 1) % batch_size == 0 and i < total_situations - 1:
            print(f"⏸️ Pause de 10 secondes après le batch {(i+1)//batch_size}...")
            time.sleep(10)
        else:
            # Petite pause entre chaque requête
            time.sleep(2)
    
    # Sauvegarde du fichier consolidé
    consolidated_filename = "tous_les_quiz_auto.json"
    with open(output_dir / consolidated_filename, "w", encoding="utf-8") as f:
        json.dump({
            "metadata": {
                "total_quiz": len(all_quizzes),
                "genere_le": "2025-10-01",
                "projet": "Complice - Quiz habiletés sociales"
            },
            "quiz": all_quizzes
        }, f, indent=2, ensure_ascii=False)
    
    print(f"\n🎉 Génération terminée !")
    print(f"📊 {len(all_quizzes)}/{total_situations} quiz générés avec succès")
    print(f"📁 Fichiers sauvegardés dans: {output_dir}")
    print(f"📄 Fichier consolidé: {consolidated_filename}")
    
    return all_quizzes

In [28]:
# 🚀 Lancement de la génération automatique !
print("🎯 Démarrage de la génération automatisée de quiz...")
print(f"📁 Dossier de sortie: {output_dir}")

# Génération de tous les quiz
tous_les_quiz = generate_all_quizzes(situations_base)

print(f"\n🎊 TERMINÉ ! {len(tous_les_quiz)} quiz générés avec succès !")

🎯 Démarrage de la génération automatisée de quiz...
📁 Dossier de sortie: generated_quizzes
🎯 Génération de 22 quiz en cours...
📦 Traitement par batch de 5 quiz

📝 Quiz 1/22: Génération en cours...
🔍 Réponse brute de GPT-4o:
'{
  "contexte": "Tu arrives en retard en cours et tu constates que tous les élèves et le professeur te regardent.",
  "question": "Que fais-tu dans cette situation ?",
  "options": {
    "A": "Je m'excuse pour le retard, je m'installe discrètement à ma place et je me concentre sur le cours.",
    "B": "J'entre bruyamment et je fais une blague pour détendre l'atmosphère.",
    "C": "Je reste à l'entrée de la classe sans rien dire, en espérant que quelqu'un viendra m'aider."
  },
  "bonne_reponse": "A",
  "commentaire": "S'excuser et s'installer discrètement montre du respect pour le professeur et les autres élèves. Cela permet aussi de reprendre le cours normalement sans trop attirer l'attention sur toi."
}'
🔍 Longueur: 715 caractères
🔍 Contenu nettoyé:
'{
  "contex

In [29]:
# 🔄 Fonction pour mélanger les réponses dans un quiz
import random
import json

def shuffle_quiz_answers(quiz_data):
    """
    Mélange les réponses de chaque question pour éviter la prévisibilité
    """
    shuffled_quiz = quiz_data.copy()
    
    for question in shuffled_quiz.get('questions', []):
        if 'options' in question and 'bonne_reponse' in question:
            options = question['options'].copy()
            correct_answer = question['bonne_reponse']
            
            # Trouver l'index de la bonne réponse
            correct_index = None
            for i, option in enumerate(options):
                if option['lettre'] == correct_answer:
                    correct_index = i
                    break
            
            if correct_index is not None:
                # Mélanger les options
                random.shuffle(options)
                
                # Réassigner les lettres A, B, C, D
                letters = ['A', 'B', 'C', 'D']
                for i, option in enumerate(options):
                    option['lettre'] = letters[i]
                    
                # Mettre à jour la bonne réponse
                for i, option in enumerate(options):
                    if option == question['options'][correct_index]:
                        question['bonne_reponse'] = letters[i]
                        break
                
                question['options'] = options
    
    return shuffled_quiz

def shuffle_all_quizzes(quizzes_list):
    """
    Mélange les réponses de tous les quiz dans une liste
    """
    shuffled_quizzes = []
    for quiz in quizzes_list:
        shuffled_quiz = shuffle_quiz_answers(quiz)
        shuffled_quizzes.append(shuffled_quiz)
    
    return shuffled_quizzes

print("✅ Fonctions de mélange des réponses créées !")

✅ Fonctions de mélange des réponses créées !


In [30]:
# 🍽️ Chargement et intégration du quiz_cantine

# Charger le quiz_cantine existant
quiz_cantine_path = output_dir / "quiz_cantine.json"

if quiz_cantine_path.exists():
    with open(quiz_cantine_path, "r", encoding="utf-8") as f:
        quiz_cantine = json.load(f)
    print("✅ Quiz cantine chargé avec succès !")
    
    # Ajouter le quiz cantine à la liste
    tous_les_quiz_avec_cantine = tous_les_quiz + [quiz_cantine]
    print(f"📊 Nombre total de quiz avec cantine: {len(tous_les_quiz_avec_cantine)}")
else:
    print("⚠️ Fichier quiz_cantine.json non trouvé, utilisation de la liste existante")
    tous_les_quiz_avec_cantine = tous_les_quiz

✅ Quiz cantine chargé avec succès !
📊 Nombre total de quiz avec cantine: 23


In [31]:
# 🎯 Application du mélange des réponses et sauvegarde finale

print("🔄 Mélange des réponses en cours...")

# Appliquer le mélange des réponses à tous les quiz (y compris cantine)
quiz_melanges = shuffle_all_quizzes(tous_les_quiz_avec_cantine)

print(f"✅ {len(quiz_melanges)} quiz avec réponses mélangées !")

# Sauvegarde de la version finale avec réponses mélangées
final_filename = "quiz_complets_melanges.json"
final_data = {
    "metadata": {
        "total_quiz": len(quiz_melanges),
        "genere_le": "2025-10-01",
        "projet": "Complice - Quiz habiletés sociales",
        "version": "finale_avec_melange",
        "description": "Quiz avec réponses mélangées pour éviter la prévisibilité + quiz cantine intégré"
    },
    "quiz": quiz_melanges
}

with open(output_dir / final_filename, "w", encoding="utf-8") as f:
    json.dump(final_data, f, indent=2, ensure_ascii=False)

print(f"🎉 TERMINÉ ! Fichier final créé: {final_filename}")
print(f"📁 Localisation: {output_dir / final_filename}")
print(f"📊 Total: {len(quiz_melanges)} quiz (22 générés + 1 cantine)")
print("🔀 Réponses mélangées pour éviter la prévisibilité !")

🔄 Mélange des réponses en cours...
✅ 23 quiz avec réponses mélangées !
🎉 TERMINÉ ! Fichier final créé: quiz_complets_melanges.json
📁 Localisation: generated_quizzes\quiz_complets_melanges.json
📊 Total: 23 quiz (22 générés + 1 cantine)
🔀 Réponses mélangées pour éviter la prévisibilité !


In [32]:
# 🔄 Fonction de mélange améliorée pour résoudre le problème
import random

def shuffle_quiz_answers_improved(quiz_data):
    """
    Version améliorée qui mélange vraiment les réponses correctement
    """
    shuffled_quiz = quiz_data.copy()
    
    if 'options' in shuffled_quiz and 'bonne_reponse' in shuffled_quiz:
        options = shuffled_quiz['options']
        current_correct = shuffled_quiz['bonne_reponse']
        
        # Extraire les réponses dans une liste
        answers = []
        letters = ['A', 'B', 'C', 'D']
        
        for letter in letters:
            if letter in options:
                answers.append({
                    'content': options[letter],
                    'was_correct': letter == current_correct
                })
        
        # Mélanger les réponses
        random.shuffle(answers)
        
        # Réassigner aux lettres A, B, C, D
        new_options = {}
        new_correct = None
        
        for i, answer in enumerate(answers):
            letter = letters[i]
            new_options[letter] = answer['content']
            if answer['was_correct']:
                new_correct = letter
        
        # Mettre à jour le quiz
        shuffled_quiz['options'] = new_options
        shuffled_quiz['bonne_reponse'] = new_correct
    
    return shuffled_quiz

def shuffle_all_quizzes_improved(quizzes_list):
    """
    Version améliorée pour mélanger tous les quiz
    """
    shuffled_quizzes = []
    for quiz in quizzes_list:
        shuffled_quiz = shuffle_quiz_answers_improved(quiz)
        shuffled_quizzes.append(shuffled_quiz)
    
    return shuffled_quizzes

print("✅ Fonctions de mélange améliorées créées !")

✅ Fonctions de mélange améliorées créées !


In [33]:
# 🎲 Test et application du mélange corrigé

print("🔄 Application du mélange amélioré...")

# Utiliser la liste existante avec cantine
quiz_vraiment_melanges = shuffle_all_quizzes_improved(tous_les_quiz_avec_cantine)

# Vérification de la distribution des bonnes réponses
distribution = {'A': 0, 'B': 0, 'C': 0, 'D': 0}
for quiz in quiz_vraiment_melanges:
    if 'bonne_reponse' in quiz:
        letter = quiz['bonne_reponse']
        if letter in distribution:
            distribution[letter] += 1

print("📊 Distribution des bonnes réponses après mélange :")
for letter, count in distribution.items():
    percentage = (count / len(quiz_vraiment_melanges)) * 100
    print(f"   {letter}: {count} quiz ({percentage:.1f}%)")

# Sauvegarde corrigée
corrected_filename = "quiz_complets_melanges_corriges.json"
corrected_data = {
    "metadata": {
        "total_quiz": len(quiz_vraiment_melanges),
        "genere_le": "2025-10-01",
        "projet": "Complice - Quiz habiletés sociales",
        "version": "finale_melange_corrige",
        "description": "Quiz avec réponses vraiment mélangées + quiz cantine intégré"
    },
    "quiz": quiz_vraiment_melanges
}

with open(output_dir / corrected_filename, "w", encoding="utf-8") as f:
    json.dump(corrected_data, f, indent=2, ensure_ascii=False)

print(f"\n🎉 CORRIGÉ ! Fichier avec mélange correct: {corrected_filename}")
print(f"📁 Localisation: {output_dir / corrected_filename}")
print("🔀 Les bonnes réponses sont maintenant vraiment mélangées !")

🔄 Application du mélange amélioré...
📊 Distribution des bonnes réponses après mélange :
   A: 11 quiz (47.8%)
   B: 9 quiz (39.1%)
   C: 3 quiz (13.0%)
   D: 0 quiz (0.0%)

🎉 CORRIGÉ ! Fichier avec mélange correct: quiz_complets_melanges_corriges.json
📁 Localisation: generated_quizzes\quiz_complets_melanges_corriges.json
🔀 Les bonnes réponses sont maintenant vraiment mélangées !


In [34]:
# 📁 Copie du fichier de quiz vers l'interface Streamlit
import shutil
from pathlib import Path

# Chemins source et destination
source_file = output_dir / "quiz_complets_melanges_corriges.json"
interface_dir = Path("../interface/data")
destination_file = interface_dir / "quiz_complets_melanges_corriges.json"

# Créer le dossier de destination s'il n'existe pas
interface_dir.mkdir(parents=True, exist_ok=True)

if source_file.exists():
    # Copier le fichier
    shutil.copy2(source_file, destination_file)
    print(f"✅ Fichier copié avec succès !")
    print(f"📂 Source: {source_file}")
    print(f"📁 Destination: {destination_file}")
    
    # Vérifier que le fichier a bien été copié
    if destination_file.exists():
        # Charger et vérifier le contenu
        with open(destination_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        total_quiz = data["metadata"]["total_quiz"]
        print(f"🎯 {total_quiz} quiz disponibles pour l'interface Streamlit")
        print(f"📊 Répartition des bonnes réponses vérifiée dans l'interface !")
    else:
        print("❌ Erreur lors de la copie")
else:
    print(f"❌ Fichier source non trouvé : {source_file}")
    print("💡 Assure-toi d'avoir exécuté les cellules de génération des quiz")

✅ Fichier copié avec succès !
📂 Source: generated_quizzes\quiz_complets_melanges_corriges.json
📁 Destination: ..\interface\data\quiz_complets_melanges_corriges.json
🎯 23 quiz disponibles pour l'interface Streamlit
📊 Répartition des bonnes réponses vérifiée dans l'interface !
